In [1]:
import pickle
import re
from itertools import pairwise

import numpy as np

from src.gpt import *

In [2]:
np.random.seed(42)

In [3]:
def extract_turns(max_turn_chars=100):
    with open(DATA_FILE, encoding="utf-8") as f:
        text = f.read()

    blocks = re.split(r"\n\s*\n", text.strip())
    turns = []

    for block in blocks:
        lines = block.strip("\n").split("\n")
        if not lines:
            continue

        m = re.compile(r"^([A-Z][A-Za-z' ]{0,30}):\s*$").match(lines[0].strip())
        if not m:
            continue

        speaker = m.group(1).strip()
        content = " ".join(line.strip() for line in lines[1:] if line.strip())
        if not content:
            continue

        turns.append((speaker, content[:max_turn_chars]))

    return turns

In [4]:
DATA_FILE = "../../tinyshakespeare.txt"
SFT_SAMPLES = "../../sft-samples.pkl"

In [5]:
CONTEXT_SIZE = 32

In [6]:
dataset = CharDataset(DATA_FILE, 1, CONTEXT_SIZE)

pairs = list(pairwise(extract_turns()))
np.random.shuffle(pairs)
pairs = pairs[:1024]

samples = []
for (speaker_a, content_a), (speaker_b, content_b) in pairs:
    prompt = f"{speaker_a}:\n{content_a}\n"
    response = f"{speaker_b}:\n{content_b}\n"
    samples.append((dataset.encode(prompt), dataset.encode(response)))

with open(SFT_SAMPLES, "wb") as f:
    pickle.dump(samples, f)
print(f"saved {len(samples)} SFT samples to {SFT_SAMPLES}")

saved 1024 SFT samples to ../../sft-samples.pkl


In [7]:
sample = samples[0]
print(f"{dataset.decode(sample[0])}\n{dataset.decode(sample[1])}")

Volsce:
You had more beard when I last saw you; but your favour is well approved by your tongue. What's the 

Roman:
There hath been in Rome strange insurrections; the people against the senators, patricians, and nobl

